# CarePath DARAG — Part 2: Train + Evaluate (GPU / L4)

Run on an **L4/GPU runtime**. Restores the Part 1 artifacts from Drive, then runs
the full DARAG method (paper *Failing Forward*, Findings of ACL 2025):

1. Synthetic in-domain transcripts (few-shot LLM, §4.1 Step 1)
2. **Voice-cloning TTS** conditioned on in-domain speakers (§4.1 Step 2 / App. D)
3. Synthetic GEC pairs via Gipformer over the cloned audio (§4.1 Step 3)
4. **Leakage report** — cosine + BLEU vs real (App. C / Table 6)
5. **QLoRA fine-tune** — full run **plus the w/o-RAC / w/o-Aug / only-Synth
   ablations** (§5)
6. Predict + **WER and NE-F1 tables** (Tables 3 & 4) + acceptance gate


In [ ]:
!nvidia-smi
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer \
  transformers accelerate peft trl bitsandbytes sentence-transformers pyvi sacrebleu
# Voice cloning (viXTTS / XTTS-v2). If this wheel is slow/unavailable, the TTS step
# falls back to single-speaker MMS (labeled tts_provider="mms_no_clone").
!pip install -q coqui-tts || pip install -q TTS

## Get the repo into Colab

In [ ]:
# Make this CarePath repo visible to Colab. Three options:
#   1. Upload carepath.zip to /content/carepath.zip via the Files sidebar.
#   2. Set CAREPATH_REPO_ZIP to a zip path in /content or Drive.
#   3. Set CAREPATH_REPO_URL to a git URL to clone.
import os, subprocess, sys, zipfile
from pathlib import Path

REPO = Path("/content/carepath")
zip_path = os.environ.get("CAREPATH_REPO_ZIP", "/content/carepath.zip")
repo_url = os.environ.get("CAREPATH_REPO_URL")

if (REPO / "pyproject.toml").exists():
    print("Repo already present at", REPO)
elif Path(zip_path).exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content/carepath_unzip")
    # the zip may contain a top-level folder; find the dir with pyproject.toml
    roots = [p.parent for p in Path("/content/carepath_unzip").rglob("pyproject.toml")]
    src = roots[0] if roots else Path("/content/carepath_unzip")
    REPO.mkdir(exist_ok=True)
    subprocess.run(f"cp -r '{src}'/* '{REPO}'/", shell=True, check=True)
    print("Unzipped repo into", REPO)
elif repo_url:
    subprocess.run(["git", "clone", repo_url, str(REPO)], check=True)
else:
    raise SystemExit("Provide carepath.zip, CAREPATH_REPO_ZIP, or CAREPATH_REPO_URL.")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "apps" / "api"))
print("cwd:", os.getcwd())

In [ ]:
# Helper: run a pipeline CLI with PYTHONPATH set, streaming output, raising on failure.
import os, subprocess, sys

def run_step(args, env_extra=None):
    env = dict(os.environ)
    env["PYTHONPATH"] = "apps/api"
    env["PYTHONIOENCODING"] = "utf-8"
    if env_extra:
        env.update(env_extra)
    print(">>>", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed ({proc.returncode}): {' '.join(args)}")

## Restore Part 1 artifacts from Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/carepath_artifacts")
DATASTORE = "artifacts/retrieval/term_datastore.json"
PAIRS = "artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl"
for rel in [DATASTORE, PAIRS]:
    dst = Path(rel); dst.parent.mkdir(parents=True, exist_ok=True)
    src = DRIVE / dst.name
    if not src.exists():
        raise SystemExit(f"Missing {src}. Run Part 1 (CPU) first.")
    shutil.copy(src, dst); print("restored", dst)

## Run size — keep smoke defaults, raise for a real run

In [ ]:
SYNTH_COUNT = 50       # synthetic transcripts to generate (paper nsyn = n)
SYNTH_TTS_LIMIT = 10   # how many to voice-clone + ASR for a smoke run
NSYN_FACTOR = 1.0      # synthetic pairs added to train = factor * |real train|
MAX_STEPS = 60         # raise (e.g. 300+) for a real fine-tune
TTS_PROVIDER = "xtts"  # "mms" for the single-speaker fallback
SYNTH_CLEAN = "artifacts/synthetic/synthetic_clean.jsonl"
TTS_MANIFEST = "artifacts/synthetic/synthetic_audio_manifest.jsonl"
SYNTH_PAIRS = "artifacts/gec_pairs/darag_synthetic_pairs.jsonl"
AUGMENTED = "artifacts/gec_pairs/darag_augmented.jsonl"
ADAPTERS = "artifacts/gec_lora/qwen3"

## 1. Synthetic in-domain transcripts (paper §4.1 Step 1)

In [ ]:
run_step([
    "scripts/gec/gen_synthetic.py",
    "--pairs", PAIRS,
    "--output", SYNTH_CLEAN,
    "--count", str(SYNTH_COUNT),
    "--load-in-4bit",
])

## 2. Voice-cloning TTS + synthetic GEC pairs (paper §4.1 Steps 2-3)

TTS is conditioned on random in-domain ViMedCSS reference clips (voice cloning).
Gipformer then transcribes the cloned audio to produce synthetic `raw_asr → gold`
pairs whose errors mimic the test-time distribution.

In [ ]:
run_step([
    "scripts/gec/voice_clone_tts.py",
    "--input", SYNTH_CLEAN,
    "--output", TTS_MANIFEST,
    "--provider", TTS_PROVIDER,
    "--ref-dataset", "tensorxt/ViMedCSS",
    "--ref-count", "20",
    "--limit", str(SYNTH_TTS_LIMIT),
    "--resume",
])
run_step([
    "scripts/gec/make_synth_pairs.py",
    "--input", TTS_MANIFEST,
    "--output", SYNTH_PAIRS,
    "--datastore", DATASTORE,
    "--resume",
])

## 3. Leakage report — synthetic is in-domain but not memorized (paper Table 6)

In [ ]:
run_step([
    "scripts/gec/check_leakage.py",
    "--synthetic", SYNTH_CLEAN,
    "--real", PAIRS,
    "--output", "artifacts/evaluations/leakage.json",
])

## 4. Augment training set + QLoRA fine-tune all variants (paper §5)

In [ ]:
run_step([
    "scripts/gec/augment.py",
    "--real", PAIRS,
    "--synthetic", SYNTH_PAIRS,
    "--output", AUGMENTED,
    "--nsyn-factor", str(NSYN_FACTOR),
])
# Trains full + wo_rac + wo_aug + only_synth into <ADAPTERS>/<variant>.
# For a fast smoke run, drop --all-variants to train just the full adapter.
run_step([
    "scripts/gec/train.py",
    "--pairs", AUGMENTED,
    "--output-dir", ADAPTERS,
    "--all-variants",
    "--max-steps", str(MAX_STEPS),
])

## 5. LLM/RAG baseline + trained predictions (paper +GEC vs +DARAG)

The LLM/RAG baseline uses the live CarePath retrieval + LLM (set `LLM_PROVIDER`,
`LLM_API_KEY` if you want CKey; otherwise it uses the offline fallback).

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "offline")
EVAL_SPLIT = AUGMENTED  # score on the frozen val/test/hard rows inside the file
run_step([
    "scripts/gec/llm_rag_baseline.py",
    "--input", PAIRS,
    "--output", "artifacts/evaluations/llm_rag.jsonl",
])
# Run the full DARAG adapter over the LLM/RAG output so one file has every column.
run_step([
    "scripts/gec/predict.py",
    "--pairs", "artifacts/evaluations/llm_rag.jsonl",
    "--adapter-dir", f"{ADAPTERS}/full",
    "--output", "artifacts/evaluations/darag_all_preds.jsonl",
    "--column", "gec_pred",
])

## 6. WER + NE-F1 tables (paper Tables 3 & 4) + acceptance gate

In [ ]:
run_step([
    "scripts/gec/evaluate.py",
    "--input", "artifacts/evaluations/darag_all_preds.jsonl",
    "--prediction-columns", "raw_asr", "corrected_text", "gec_pred",
    "--wer-output", "artifacts/evaluations/darag_wer.json",
    "--ne-f1-output", "artifacts/evaluations/darag_ne_f1.json",
])
# Gate: trained adapter must match-or-beat raw + LLM/RAG on val+hard (non-zero on REJECT).
run_step(["scripts/gec/gate.py", "--report", "artifacts/evaluations/darag_wer.json"])

## 7. Save adapters + metrics to Google Drive

In [ ]:
import shutil
from pathlib import Path
out = DRIVE / "part2"
out.mkdir(parents=True, exist_ok=True)
for rel in [
    "artifacts/evaluations/darag_wer.json",
    "artifacts/evaluations/darag_ne_f1.json",
    "artifacts/evaluations/leakage.json",
]:
    p = Path(rel)
    if p.exists():
        shutil.copy(p, out / p.name); print("saved", out / p.name)
adapters = Path(ADAPTERS)
if adapters.exists():
    dst = out / "gec_lora"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(adapters, dst); print("saved adapters ->", dst)
print("Part 2 done.")